# Composition-Engineering (Pipeline Step 1f)

Extracted from `Pre_DE_processing.ipynb` into its own pipeline step. Loads the
fully-annotated cohort, restricts to the analysis cohort (drop Olfactory, keep
MCI), tags the composition-engineering scenarios, and exports the single cohort
(with the `G1_*`/`G2_*` masks) for the DE scripts.

Every experiment parameter — the contrast column, the two groups, the deviation
magnitudes, and the per-method iteration counts — comes from the `composition`
block in `config.yaml`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
# Use the ezy_seq bundled with THIS repo (EzySeq_Library/mypythonlibrary/src), NOT
# the pip-installed copy. The editable install points at a different, OLDER ezy_seq
# (Carter Woods/Vignette_project) that lacks tag_region_abundance_by_FMT and the
# observed=True groupby fix. Walk up from the CWD to find the repo's library and put
# it first on sys.path so `import ezy_seq` resolves to it. (Mirrors Pre_DE_processing.)
for _d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    _ezy_src = _d / "EzySeq_Library" / "mypythonlibrary" / "src"
    if _ezy_src.is_dir():
        sys.path.insert(0, str(_ezy_src))
        break

# ── Load pipeline paths + composition config from config.yaml (per framework) ──
import ezy_seq as ezy
print("ezy_seq from:", ezy.__file__)   # MUST be the repo's EzySeq_Library copy
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os
import yaml
from pathlib import Path

def _find_config(name="config.yaml"):
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found in {Path.cwd()} or its parents")

def _select_framework(cfg, name=None):
    if "frameworks" not in cfg:            # flat/legacy config
        return cfg
    name = name or os.environ.get("PIPELINE_FRAMEWORK") or cfg.get("active_framework")
    if name not in cfg["frameworks"]:
        raise KeyError(f"Framework {name!r} not in config. Available: {list(cfg['frameworks'])}")
    print(f"Framework: {name}")
    return cfg["frameworks"][name]

def _abs_paths(section, base):
    # config.yaml paths are relative TO THE CONFIG'S DIRECTORY (repo root), not to
    # the kernel's working dir (this notebook lives in Pre_processing/). Make each
    # relative path absolute against `base`. Keys ending in "_col" are column
    # names, not paths, so leave those (and any absolute value) untouched.
    out = {}
    for k, v in section.items():
        if isinstance(v, str) and not k.endswith("_col") and not Path(v).is_absolute():
            out[k] = str((base / v).resolve())
        else:
            out[k] = v
    return out

_CONFIG_PATH = _find_config().resolve()
_CONFIG_DIR = _CONFIG_PATH.parent          # repo root — paths resolve against this
cfg = yaml.safe_load(open(_CONFIG_PATH))

# Which framework/dataset to run. MUST match the framework you preprocessed (the
# adata_h5ad this reads is per-framework). Override via env PIPELINE_FRAMEWORK,
# or set FRAMEWORK here ("genotype" or "fmt"); None -> config active_framework.
FRAMEWORK = 'fmt'
_fw = _select_framework(cfg, FRAMEWORK)
INPUTS  = _abs_paths(_fw["inputs"],  _CONFIG_DIR)
OUTPUTS = _abs_paths(_fw["outputs"], _CONFIG_DIR)
COMP    = _fw["composition"]

c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
c:\Users\woods\miniconda3\envs\new_env\lib\site-packages\anndata\utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


ezy_seq from: c:\Users\woods\OneDrive - University of Missouri\General - Lin Brain Lab - Ogrp\Iscience_Manuscript\AnatomicConfounds-Corrections\EzySeq_Library\mypythonlibrary\src\ezy_seq\__init__.py
Framework: genotype


In [7]:
OUTPUTS["adata_h5ad"]

'C:\\Users\\woods\\OneDrive - University of Missouri\\General - Lin Brain Lab - Ogrp\\Iscience_Manuscript\\AnatomicConfounds-Corrections\\Final_Data\\genotype\\adata_full.h5ad'

In [2]:
adata_full = sc.read_h5ad(OUTPUTS["adata_h5ad"])

In [ ]:
# ── Load the fully-annotated cohort and restrict to the analysis cohort ──────
# Pre_DE_processing.ipynb writes the assembled / annotated / normalized object here.
adata_full = sc.read_h5ad(OUTPUTS["adata_h5ad"])

# Framework-driven restriction (from config.yaml -> composition):
#   drop_regions  : regions removed from the cohort (e.g. Olfactory bulb).
#   cohort_filter : {obs_col: [allowed, ...]} kept before the contrast.
#                   genotype -> {FMT: [MCI]} (genotype within disease);
#                   fmt      -> {} (no restriction; keeps the Cntrl arm too).
for _region in COMP.get("drop_regions", []):
    adata_full = adata_full[adata_full.obs['napari_region'] != _region]
for _col, _allowed in (COMP.get("cohort_filter") or {}).items():
    adata_full = adata_full[adata_full.obs[_col].isin(list(_allowed))]
adata_full = adata_full.copy()

print(adata_full)
print(adata_full.obs[COMP["group_col"]].value_counts())

In [ ]:


def export_anndata_for_seurat(
    adata,
    output_dir,
    *,
    counts_layer_candidates=("counts", "raw"),
    x_filename="normalized_counts.csv",
    counts_filename="raw_counts.csv",
    var_filename="features_counts.csv",
    obs_filename="cell_metadata.csv",
    coords_key="spatial_fov",
    coords_filename="coords_xy.csv",
    pca_key="X_pca",
    pca_filename="pca.csv",
    umap_key="X_umap",
    umap_filename="umap.csv",
    export_other_obsm=True,
    other_obsm_exclude=("spatial_fov", "X_pca", "X_umap"),
    verbose=True,
):
    """
    Export an AnnData object to CSVs for reconstruction in R/Seurat.

    Writes normalized_counts.csv, raw_counts.csv, features_counts.csv,
    cell_metadata.csv, coords_xy.csv, pca.csv, umap.csv, and any other
    obsm matrices as <key>.csv.  Returns a dict of written file paths.
    """
    import os
    import numpy as np
    import pandas as pd
    from scipy import sparse

    os.makedirs(output_dir, exist_ok=True)
    written = {}

    def _dense(mat):
        if sparse.issparse(mat): return mat.toarray()
        if hasattr(mat, "toarray"): return mat.toarray()
        return np.asarray(mat)

    # Raw counts
    layer = next((l for l in counts_layer_candidates if l in getattr(adata, "layers", {})), None)
    counts = adata.layers[layer] if layer else adata.X
    if layer is None and verbose:
        print("Warning: no counts layer found; using X for raw counts")
    fp = os.path.join(output_dir, counts_filename)
    pd.DataFrame(_dense(counts), index=adata.obs_names, columns=adata.var_names).to_csv(fp)
    written["counts"] = fp

    # Normalized (X)
    fp = os.path.join(output_dir, x_filename)
    pd.DataFrame(_dense(adata.X), index=adata.obs_names, columns=adata.var_names).to_csv(fp)
    written["X"] = fp

    # Metadata
    for name, fname in [(adata.var, var_filename), (adata.obs, obs_filename)]:
        fp = os.path.join(output_dir, fname)
        name.to_csv(fp)
        written[fname] = fp

    # Spatial coords
    if coords_key in getattr(adata, "obsm", {}):
        fp = os.path.join(output_dir, coords_filename)
        coords = np.asarray(adata.obsm[coords_key])[:, :2]
        pd.DataFrame(coords, index=adata.obs_names, columns=["x", "y"]).to_csv(fp)
        written["coords"] = fp

    # PCA / UMAP
    for key, fname, col_fmt in [(pca_key, pca_filename, "PC{}"), (umap_key, umap_filename, "UMAP_{}")]:
        if key in getattr(adata, "obsm", {}):
            arr = np.asarray(adata.obsm[key])
            cols = [col_fmt.format(i + 1) for i in range(arr.shape[1])]
            fp = os.path.join(output_dir, fname)
            pd.DataFrame(arr, index=adata.obs_names, columns=cols).to_csv(fp)
            written[key] = fp

    # Any other obsm
    if export_other_obsm:
        for key in adata.obsm_keys():
            if key in other_obsm_exclude: continue
            arr = np.asarray(adata.obsm[key])
            if arr.ndim != 2 or arr.shape[0] != adata.n_obs: continue
            fp = os.path.join(output_dir, f"{key}.csv")
            pd.DataFrame(arr, index=adata.obs_names,
                         columns=[f"{key}_{i+1}" for i in range(arr.shape[1])]).to_csv(fp)
            written[f"obsm:{key}"] = fp

    if verbose:
        print(f"Export complete: {len(written)} files written to {output_dir}")
    return written


In [ ]:
# ── Composition-engineering (config-driven) ────────────────────────
# Contrast COMP['group_col'] (Genotype: up_group vs down_group) within the MCI
# cohort. For every deviation magnitude, tag deseq2_iterations independent
# iterations (random_state 0..N-1) of the two opposite cortex-shift scenarios:
#   G1_{seed}_{dev} = up_group cortex-high / down_group cortex-low
#   G2_{seed}_{dev} = the reverse
# The DE scripts consume these masks: DESeq2 uses all iterations per dev, the
# Dream LMM uses only the first COMP['dream_iterations'].
# match_scenarios (from config) size-matches each sample between G1 and G2: a given
# sample gets the same cell count in both mirror scenarios (counts still vary across
# samples). match_reference_dev additionally holds those per-sample counts fixed
# across deviations at the reference (most extreme) dev. See tag_region_abundance_by_FMT.
# NOTE: tag_region_abundance_by_FMT keeps its legacy 'fmt_col'/'up_fmt'/'down_fmt'
# argument names; we drive them from the config's group_col/up_group/down_group.
for dev_mag in COMP["deviations"]:
    for r_state in range(COMP["deseq2_iterations"]):
        ezy.tag_region_abundance_by_FMT(
            adata_full,
            dev=dev_mag,
            random_state=r_state,
            balance=COMP["balance"],
            match_scenarios=COMP.get("match_scenarios", False),
            match_reference_dev=COMP.get("match_reference_dev"),
            region_of_interest=COMP["region_of_interest"],
            fmt_col=COMP["group_col"],
            up_fmt=COMP["up_group"],
            down_fmt=COMP["down_group"],
        )

In [ ]:
# ── QC: between-group region composition WITHIN each scenario (G1, then G2) ──────
# For a chosen (seed, dev), split each scenario's cells by treatment group
# (COMP['group_col'], e.g. FMT: Stroke_FMT vs Healthy_FMT), give each group's
# per-region fractions, and report their difference. The engineered confound: in
# the region_of_interest the between-group difference is large and FLIPS SIGN from
# G1 to G2 (up-group cortex-high in G1, cortex-low in G2), while pooled G1 vs G2
# stay matched.
import matplotlib.pyplot as plt

def compare_region_composition(adata, g1_col, g2_col, region_col="napari_region",
                               group_col=None, groups=None, plot=True):
    """
    Compare the two treatment groups' region composition WITHIN each scenario.

    Runs the between-group comparison once for G1 and once for G2. Within a
    scenario, cells carrying that mask are split by `group_col` (default
    COMP['group_col']); each group's cells are normalised to sum to 1 over regions,
    and 'difference' = groups[0] - groups[1] (up_group - down_group, e.g.
    Stroke_FMT - Healthy_FMT).

    Parameters
    ----------
    g1_col, g2_col : str
        The two scenario mask columns (e.g. 'G1_0_1.5' / 'G2_0_1.5').
    group_col, groups : str, tuple
        Treatment column and the two levels to contrast; default to the COMP config
        (group_col, up_group vs down_group).

    Returns
    -------
    dict[str, pd.DataFrame]
        {'G1': df, 'G2': df}; each indexed by region with one fraction column per
        group plus 'difference' (groups[0] - groups[1]).
    """
    group_col = group_col or COMP["group_col"]
    if groups is None:
        groups = (COMP["up_group"], COMP["down_group"])
    roi = COMP.get("region_of_interest")

    def _between_groups(mask_col):
        if mask_col not in adata.obs:
            raise KeyError(f"{mask_col!r} not found in adata.obs (run the tagging cell first)")
        sub = adata.obs[adata.obs[mask_col].astype(bool)]
        ct = sub.groupby([group_col, region_col], observed=True).size().unstack(fill_value=0)
        comp = ct.div(ct.sum(axis=1), axis=0).T.reindex(columns=list(groups)).fillna(0.0)
        comp = comp.sort_values(groups[0], ascending=False)
        comp["difference"] = comp[groups[0]] - comp[groups[1]]
        return comp

    out = {"G1": _between_groups(g1_col), "G2": _between_groups(g2_col)}

    for name, col in (("G1", g1_col), ("G2", g2_col)):
        comp = out[name]
        print(f"\n{name} ({col}) — region composition by {group_col} "
              f"[{groups[0]} vs {groups[1]}]; difference = {groups[0]} - {groups[1]}:")
        print(comp.round(4).to_string())
        if roi in comp.index:
            print(f"  {roi} difference ({groups[0]} - {groups[1]}): {comp.loc[roi, 'difference']:+.4f}")
    if roi in out["G1"].index and roi in out["G2"].index:
        print(f"\n{roi} between-group difference flips:  "
              f"G1 {out['G1'].loc[roi, 'difference']:+.4f}  ->  G2 {out['G2'].loc[roi, 'difference']:+.4f}")

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
        for ax, name in zip(axes, ("G1", "G2")):
            out[name][list(groups)].plot(kind="bar", ax=ax)
            ax.set_title(f"{name}: {groups[0]} vs {groups[1]}")
            ax.set_xlabel(region_col); ax.set_ylabel("fraction of group's cells")
            ax.legend(title=group_col)
        plt.tight_layout(); plt.show()
    return out

# Between-group (up vs down) region differences within G1, then within G2, for one
# (seed, dev). The region_of_interest difference should flip sign between them:
comp_scenarios = compare_region_composition(adata_full, "G1_0_1.5", "G2_0_1.5")

In [6]:
adata_full

AnnData object with n_obs × n_vars = 359526 × 1000
    obs: 'cell_type', 'RNA_E2FAD_Cell.Typing.InSituType.1_1_posterior_probability', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.Histone', 'Max.Histone', 'Mean.G', 'Max.G', 'Mean.rRNA', 'Max.rRNA', 'Mean.GFAP', 'Max.GFAP', 'Mean.DAPI', 'Max.DAPI', 'SplitRatioToLocal', 'NucArea', 'NucAspectRatio', 'Circularity', 'Eccentricity', 'Perimeter', 'Solidity', 'cell_id', 'assay_type', 'version', 'Run_Tissue_name', 'Panel', 'cellSegmentationSetId', 'cellSegmentationSetName', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 

In [5]:
adata_full.obs['G1_0_1.5']

KeyError: 'G1_0_1.5'

In [ ]:
# ── Export the tagged full cohort for the DE scripts ──────────────────
# The G1_*/G2_* masks ride along in cell_metadata.csv; DE_Base_Analysis uses the
# whole cohort while LMM_all reconstructs the per-iteration cortex-up/down subsets
# from the masks.
written_files = export_anndata_for_seurat(adata_full, OUTPUTS["de_export_base"])